In [1]:
import os
from pathlib import Path

cwd = Path.cwd()

if cwd.name == "notebooks":
    os.chdir(cwd.parent)

In [ ]:
from src.models.vit_backbones import *

def _shape_of(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None

def check_same_architecture(m1, m2, *, ignore=("classifier", "head", "score")):
    c1, c2 = m1.config, m2.config

    vit_keys = [
        "model_type",
        "hidden_size",
        "num_hidden_layers",
        "num_attention_heads",
        "intermediate_size",
        "image_size",
        "patch_size",
        "num_channels",
        "qkv_bias",
        "hidden_act",
        "layer_norm_eps",
    ]

    cfg_diff = {}
    for k in vit_keys:
        v1 = getattr(c1, k, None)
        v2 = getattr(c2, k, None)
        if v1 != v2:
            cfg_diff[k] = (v1, v2)

    sd1 = m1.state_dict()
    sd2 = m2.state_dict()

    def keep_key(k: str) -> bool:
        return not any(tok in k for tok in ignore)

    keys1 = {k for k in sd1.keys() if keep_key(k)}
    keys2 = {k for k in sd2.keys() if keep_key(k)}

    missing_in_2 = sorted(keys1 - keys2)
    missing_in_1 = sorted(keys2 - keys1)

    # Compare shapes for common keys
    common = sorted(keys1 & keys2)
    shape_mismatch = []
    for k in common:
        s1, s2 = _shape_of(sd1[k]), _shape_of(sd2[k])
        if s1 != s2:
            shape_mismatch.append((k, s1, s2))

    same_cfg = len(cfg_diff) == 0
    same_keys = (len(missing_in_1) == 0 and len(missing_in_2) == 0)
    same_shapes = len(shape_mismatch) == 0

    return {
        "same_config_on_vit_keys": same_cfg,
        "config_differences": cfg_diff,
        "same_state_dict_keys_(ignoring_heads)": same_keys,
        "missing_keys_in_expression_model": missing_in_2,
        "missing_keys_in_imagenet_model": missing_in_1,
        "same_tensor_shapes_(common_keys)": same_shapes,
        "shape_mismatches": shape_mismatch,
        "num_common_keys_compared": len(common),
    }

@torch.no_grad()
def compare_weights(m1, m2, *, ignore=("classifier", "head", "score"), atol=0.0, rtol=0.0):
    sd1 = m1.state_dict()
    sd2 = m2.state_dict()

    def keep_key(k: str) -> bool:
        return not any(tok in k for tok in ignore)

    keys1 = {k for k in sd1.keys() if keep_key(k)}
    keys2 = {k for k in sd2.keys() if keep_key(k)}
    common = sorted(keys1 & keys2)

    diffs = []
    identical = 0
    skipped = 0

    for k in common:
        t1 = sd1[k]
        t2 = sd2[k]

        if t1.shape != t2.shape:
            skipped += 1
            continue

        t1c = t1.detach().float().cpu()
        t2c = t2.detach().float().cpu()

        delta = (t1c - t2c).abs()
        max_abs = delta.max().item() if delta.numel() else 0.0
        mean_abs = delta.mean().item() if delta.numel() else 0.0
        l2 = torch.norm((t1c - t2c).reshape(-1), p=2).item() if delta.numel() else 0.0

        same = torch.allclose(t1c, t2c, atol=atol, rtol=rtol)
        if same:
            identical += 1
        else:
            diffs.append({
                "key": k,
                "shape": tuple(t1.shape),
                "max_abs_diff": max_abs,
                "mean_abs_diff": mean_abs,
                "l2_norm_diff": l2,
            })

    diffs_sorted = sorted(diffs, key=lambda d: d["max_abs_diff"], reverse=True)

    return {
        "compared_keys": len(common),
        "identical_keys": identical,
        "different_keys": len(diffs_sorted),
        "skipped_due_to_shape_mismatch": skipped,
        "different_layers_sorted": diffs_sorted,
    }


/home/tymek/studies_AI/ViT-Empathy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

imagenet = load_imagenet_vit(device=device)
expr = load_expression_vit(device=device)
arch_report = check_same_architecture(imagenet, expr, ignore=("classifier", "head", "score"))
print("\n=== ARCHITECTURE REPORT (ignoring heads) ===")
for k, v in arch_report.items():
    if isinstance(v, list) and len(v) > 10:
        print(f"{k}: {len(v)} items (showing first 10)")
        for item in v[:10]:
            print("  ", item)
    else:
        print(f"{k}: {v}")
head_arch_report = check_same_architecture(imagenet, expr, ignore=())
print("\n=== ARCHITECTURE REPORT (including heads) ===")
print("same_state_dict_keys_(including_heads):", head_arch_report["same_state_dict_keys_(ignoring_heads)"])
if head_arch_report["missing_keys_in_expression_model"][:5]:
    print("missing in expr (first 5):", head_arch_report["missing_keys_in_expression_model"][:5])
if head_arch_report["missing_keys_in_imagenet_model"][:5]:
    print("missing in imagenet (first 5):", head_arch_report["missing_keys_in_imagenet_model"][:5])

weight_report = compare_weights(imagenet, expr, ignore=("classifier", "head", "score"), atol=0.0, rtol=0.0)
print("\n=== WEIGHT REPORT (ignoring heads) ===")
print("compared_keys:", weight_report["compared_keys"])
print("identical_keys:", weight_report["identical_keys"])
print("different_keys:", weight_report["different_keys"])

print("\nTop 20 differing tensors by max_abs_diff:")
for row in weight_report["different_layers_sorted"][:20]:
    print(f"{row['key']:70s} shape={row['shape']} "
          f"max={row['max_abs_diff']:.6g} mean={row['mean_abs_diff']:.6g} l2={row['l2_norm_diff']:.6g}")

head_weight_report = compare_weights(imagenet, expr, ignore=(), atol=0.0, rtol=0.0)
print("\n=== WEIGHT REPORT (including heads) ===")
print("compared_keys:", head_weight_report["compared_keys"])
print("identical_keys:", head_weight_report["identical_keys"])
print("different_keys:", head_weight_report["different_keys"])


=== ARCHITECTURE REPORT (ignoring heads) ===
same_config_on_vit_keys: True
config_differences: {}
same_state_dict_keys_(ignoring_heads): False
missing_keys_in_expression_model: 200 items (showing first 10)
   embeddings.cls_token
   embeddings.patch_embeddings.projection.bias
   embeddings.patch_embeddings.projection.weight
   embeddings.position_embeddings
   encoder.layer.0.attention.attention.key.bias
   encoder.layer.0.attention.attention.key.weight
   encoder.layer.0.attention.attention.query.bias
   encoder.layer.0.attention.attention.query.weight
   encoder.layer.0.attention.attention.value.bias
   encoder.layer.0.attention.attention.value.weight
missing_keys_in_imagenet_model: 198 items (showing first 10)
   vit.embeddings.cls_token
   vit.embeddings.patch_embeddings.projection.bias
   vit.embeddings.patch_embeddings.projection.weight
   vit.embeddings.position_embeddings
   vit.encoder.layer.0.attention.attention.key.bias
   vit.encoder.layer.0.attention.attention.key.weight


In [ ]:
import torch
from transformers import ViTModel, ViTForImageClassification

def load_imagenet_vit(device="cuda"):
    return ViTModel.from_pretrained(
        "google/vit-base-patch16-224-in21k",
        output_hidden_states=True
    ).to(device)

def load_expression_vit(device="cuda"):
    return ViTForImageClassification.from_pretrained(
        "trpakov/vit-face-expression",
        output_hidden_states=True
    ).to(device)

def vit_backbone(model):
    """Return the ViT backbone module (works for ViTModel and ViTForImageClassification)."""
    return model.vit if hasattr(model, "vit") else model

def normalize_state_dict_keys(sd, strip_prefixes=("vit.",)):
    out = {}
    for k, v in sd.items():
        nk = k
        for p in strip_prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out

def config_arch_check(m1, m2):
    c1, c2 = m1.config, m2.config
    keys = [
        "model_type", "hidden_size", "num_hidden_layers", "num_attention_heads",
        "intermediate_size", "image_size", "patch_size", "num_channels",
        "qkv_bias", "hidden_act", "layer_norm_eps"
    ]
    diffs = {k: (getattr(c1, k, None), getattr(c2, k, None))
             for k in keys if getattr(c1, k, None) != getattr(c2, k, None)}
    return diffs

@torch.no_grad()
def compare_state_dicts(sd1, sd2, *, atol=0.0, rtol=0.0):
    keys1, keys2 = set(sd1.keys()), set(sd2.keys())
    common = sorted(keys1 & keys2)
    missing_2 = sorted(keys1 - keys2)
    missing_1 = sorted(keys2 - keys1)

    shape_mismatch = []
    diffs = []
    identical = 0

    for k in common:
        t1, t2 = sd1[k], sd2[k]
        if t1.shape != t2.shape:
            shape_mismatch.append((k, tuple(t1.shape), tuple(t2.shape)))
            continue

        a = t1.detach().float().cpu()
        b = t2.detach().float().cpu()

        same = torch.allclose(a, b, atol=atol, rtol=rtol)
        if same:
            identical += 1
            continue

        delta = (a - b).abs()
        diffs.append({
            "key": k,
            "shape": tuple(t1.shape),
            "max_abs_diff": float(delta.max().item()) if delta.numel() else 0.0,
            "mean_abs_diff": float(delta.mean().item()) if delta.numel() else 0.0,
            "l2_norm_diff": float(torch.norm((a - b).reshape(-1), p=2).item()) if delta.numel() else 0.0,
        })

    diffs.sort(key=lambda d: d["max_abs_diff"], reverse=True)

    return {
        "num_keys_sd1": len(keys1),
        "num_keys_sd2": len(keys2),
        "num_common": len(common),
        "num_identical": identical,
        "num_different": len(diffs),
        "missing_in_sd2": missing_2,
        "missing_in_sd1": missing_1,
        "shape_mismatches": shape_mismatch,
        "diffs_sorted": diffs,
    }

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    imagenet = load_imagenet_vit(device=device)
    expr = load_expression_vit(device=device)

    cfg_diffs = config_arch_check(imagenet, expr)
    print("\n=== CONFIG ARCH CHECK ===")
    print("Same on key ViT hyperparams?" , (len(cfg_diffs) == 0))
    if cfg_diffs:
        for k, (a, b) in cfg_diffs.items():
            print(f"{k}: {a} vs {b}")

    b1 = vit_backbone(imagenet)
    b2 = vit_backbone(expr)

    sd1 = normalize_state_dict_keys(b1.state_dict(), strip_prefixes=()) 
    sd2 = normalize_state_dict_keys(b2.state_dict(), strip_prefixes=("vit.",))

    print("\n=== BACKBONE KEY PREFIX EXAMPLE ===")
    print("imagenet sample key:", next(iter(sd1.keys())))
    print("expr sample key    :", next(iter(sd2.keys())))

    rep = compare_state_dicts(sd1, sd2, atol=0.0, rtol=0.0)

    print("\n=== BACKBONE WEIGHT COMPARISON ===")
    print("common:", rep["num_common"])
    print("identical:", rep["num_identical"])
    print("different:", rep["num_different"])
    print("shape_mismatches:", len(rep["shape_mismatches"]))
    print("missing_in_expr:", len(rep["missing_in_sd2"]))
    print("missing_in_imagenet:", len(rep["missing_in_sd1"]))

    print("\nTop 20 backbone tensors by max_abs_diff:")
    for row in rep["diffs_sorted"][:20]:
        print(f"{row['key']:70s} shape={row['shape']} "
              f"max={row['max_abs_diff']:.6g} mean={row['mean_abs_diff']:.6g} l2={row['l2_norm_diff']:.6g}")

    print("\n=== CLASSIFIER HEAD CHECK ===")
    if hasattr(expr, "classifier"):
        head_sd = expr.classifier.state_dict()
        print("expr classifier keys:", list(head_sd.keys()))
        print("expr classifier weight shape:", tuple(expr.classifier.weight.shape))
    else:
        print("No classifier attribute found on expression model (unexpected for ViTForImageClassification).")

main()



=== CONFIG ARCH CHECK ===
Same on key ViT hyperparams? True

=== BACKBONE KEY PREFIX EXAMPLE ===
imagenet sample key: embeddings.cls_token
expr sample key    : embeddings.cls_token

=== BACKBONE WEIGHT COMPARISON ===
common: 198
identical: 0
different: 198
shape_mismatches: 0
missing_in_expr: 2
missing_in_imagenet: 0

Top 20 backbone tensors by max_abs_diff:
layernorm.weight                                                       shape=(768,) max=0.0624734 mean=0.0361461 l2=1.03032
encoder.layer.11.output.dense.weight                                   shape=(768, 3072) max=0.0599655 mean=0.00423523 l2=8.81136
encoder.layer.10.output.dense.weight                                   shape=(768, 3072) max=0.0520021 mean=0.00488513 l2=9.76005
encoder.layer.9.intermediate.dense.weight                              shape=(3072, 768) max=0.0469224 mean=0.00492214 l2=9.64703
encoder.layer.9.output.dense.weight                                    shape=(768, 3072) max=0.0425656 mean=0.00512627 l2=10

In [3]:
import torch
from src.models.vit_backbones import *

device = "cuda" if torch.cuda.is_available() else "cpu"

imagenet = load_imagenet_vit(device=device)

In [4]:
imagenet

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

In [6]:
import inspect
print(inspect.getsource(imagenet.encoder.forward))

    def forward(self, hidden_states: torch.Tensor, head_mask: Optional[torch.Tensor] = None) -> BaseModelOutput:
        for i, layer_module in enumerate(self.layer):
            layer_head_mask = head_mask[i] if head_mask is not None else None
            hidden_states = layer_module(hidden_states, layer_head_mask)

        return BaseModelOutput(last_hidden_state=hidden_states)



In [7]:
print(type(imagenet))


<class 'transformers.models.vit.modeling_vit.ViTModel'>


In [8]:
import inspect
from transformers.models.vit.modeling_vit import ViTModel

print(inspect.getsource(ViTModel.forward))


    @check_model_inputs(tie_last_hidden_states=False)
    @auto_docstring
    def forward(
        self,
        pixel_values: Optional[torch.Tensor] = None,
        bool_masked_pos: Optional[torch.BoolTensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        interpolate_pos_encoding: Optional[bool] = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> BaseModelOutputWithPooling:
        r"""
        bool_masked_pos (`torch.BoolTensor` of shape `(batch_size, num_patches)`, *optional*):
            Boolean masked positions. Indicates which patches are masked (1) and which aren't (0).
        """

        if pixel_values is None:
            raise ValueError("You have to specify pixel_values")

        # Prepare head mask if needed
        # 1.0 in head_mask indicate we keep the head
        # attention_probs has shape bsz x n_heads x N x N
        # input head_mask has shape [num_heads] or [num_hidden_layers x num_heads]
        # and head_mask is converted to s

In [9]:
from transformers.models.vit.modeling_vit import ViTEncoder

print(inspect.getsource(ViTEncoder.forward))


    def forward(self, hidden_states: torch.Tensor, head_mask: Optional[torch.Tensor] = None) -> BaseModelOutput:
        for i, layer_module in enumerate(self.layer):
            layer_head_mask = head_mask[i] if head_mask is not None else None
            hidden_states = layer_module(hidden_states, layer_head_mask)

        return BaseModelOutput(last_hidden_state=hidden_states)



In [10]:
from transformers.models.vit.modeling_vit import ViTLayer

print(inspect.getsource(ViTLayer.forward))


    def forward(self, hidden_states: torch.Tensor, head_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        hidden_states_norm = self.layernorm_before(hidden_states)
        attention_output = self.attention(hidden_states_norm, head_mask)

        # first residual connection
        hidden_states = attention_output + hidden_states

        # in ViT, layernorm is also applied after self-attention
        layer_output = self.layernorm_after(hidden_states)
        layer_output = self.intermediate(layer_output)

        # second residual connection is done here
        layer_output = self.output(layer_output, hidden_states)

        return layer_output



In [11]:
from transformers.models.vit.modeling_vit import ViTAttention

print(inspect.getsource(ViTAttention.forward))


    def forward(self, hidden_states: torch.Tensor, head_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        self_attn_output, _ = self.attention(hidden_states, head_mask)
        output = self.output(self_attn_output, hidden_states)
        return output



In [ ]:
import torch
from transformers import ViTModel

model = imagenet
device = next(model.parameters()).device
dummy_input = torch.randn(1, 3, 224, 224, device=device)

with torch.no_grad():
    outputs = model(dummy_input, output_hidden_states=True)

print("device:", device)
print("last_hidden_state:", outputs.last_hidden_state.shape)
print("pooler_output:", outputs.pooler_output.shape)
print(outputs)



device: cuda:0
last_hidden_state: torch.Size([1, 197, 768])
pooler_output: torch.Size([1, 768])
BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.2443,  0.0791,  0.0691,  ..., -0.0912, -0.0966, -0.1160],
         [ 0.1010, -0.0911, -0.0662,  ...,  0.1678, -0.0848, -0.1176],
         [ 0.3366,  0.0889, -0.1111,  ..., -0.0584,  0.0936,  0.0144],
         ...,
         [ 0.2305,  0.0224,  0.0149,  ...,  0.0842,  0.0651, -0.1432],
         [ 0.2047, -0.0668, -0.0569,  ...,  0.0981, -0.0030, -0.1632],
         [ 0.0873, -0.0820,  0.0782,  ...,  0.0735, -0.1260, -0.1427]]],
       device='cuda:0'), pooler_output=tensor([[ 3.6940e-01, -4.9233e-02, -6.0430e-01,  8.3292e-02,  9.6214e-02,
          6.6816e-01,  1.9112e-01,  2.5813e-01, -4.9384e-01,  2.1779e-01,
         -3.9923e-01,  6.9043e-01, -4.8572e-01,  2.1373e-01, -2.4982e-01,
          6.7332e-01, -7.7107e-01,  1.5765e-01, -9.2396e-02,  1.2682e-01,
          5.2624e-01, -6.8962e-01,  1.2880e-02, -5.3047e-01,  8.5271e-01,
        